Author: Mike Bollinger
based on a framework by Logan Kline
With editing and formatting support from ChatGPT v4.0
Created: 12/2/2024
Integrates Survey123 with Smartsheet to update Smartsheet rows with S123 input

In [101]:
import smartsheet
import pandas as pd
from arcgis.gis import GIS
from datetime import datetime

Start by pulling down the Survey123 data and formatting it as it'll need to be

In [ ]:
print("ArcGIS Online Org account")
gis = GIS('home')
print("Logged in to " + str(gis.properties.name) + " as " + str(gis.properties.user.username))

Survey 123 configuration and formatting to fit Smartsheet format

In [ ]:
#access and save layers from AGOL
s123_divedata = gis.content.get('ca90818eee7343f88e9ff02d0f0a9dd3').layers[1] #NCCOS UMS AUV database point layer

#convert to databases
s123_divedata_dbf = pd.DataFrame(s123_divedata.query().sdf)

#s123_divedata_dbf.head()

In [ ]:
# Extract 'x' and 'y' values from the SHAPE column and add them to the existing DataFrame
s123_divedata_dbf[['x', 'y']] = s123_divedata_dbf['SHAPE'].apply(lambda shape: pd.Series([shape['x'], shape['y']]) if isinstance(shape, dict) else pd.Series([None, None]))

#s123_divedata_dbf.head()

In [ ]:
#Subset survey123 database into fields that smartsheet accepts
s123_divedata_dbf_sub = s123_divedata_dbf.loc[:,("globalid","Project","CruiseID","StudyRegion","VesselPlatform","Vehicle","Pilot","Crew","DeployID","SiteID","DeployDate","DeployTime","y","x","SurfaceTime","DeploymentNotes")]

s123_divedata_dbf_sub = s123_divedata_dbf_sub.reset_index(drop=True)

#s123_divedata_dbf_sub.head()

In [ ]:
#Rename columns to match Smartsheet
s123_divedata_dbf_reset = s123_divedata_dbf_sub.rename(columns={"globalid":"globalid","Project": "Project", "CruiseID":"Mission","StudyRegion":"Location","VesselPlatform":"Platform","Vehicle": "Vehicle","Pilot":"Pilot","Crew":"Team","DeployDate":"Dive Date","DeployID":"Dive Number","SiteID":"Site","DeployTime":"Mission Start Time","y":"Start Latitude","x":"Start Longitude","SurfaceTime":"End Time","DeploymentNotes":"Dive Description"})

s123_divedata_dbf_reset.reset_index(drop=True, inplace=True)

s123_divedata_dbf_reset.head()

In [ ]:
##Cleaning up some smaller formatting issues to fine tune the export to smartsheet
# Convert "Mission Start Time" and "End Time" to HH:MM format
s123_divedata_dbf_reset['Mission Start Time'] = pd.to_datetime(s123_divedata_dbf_reset['Mission Start Time'], format='%H:%M:%S:%f').dt.strftime('%H:%M')
s123_divedata_dbf_reset['End Time'] = pd.to_datetime(s123_divedata_dbf_reset['End Time'], format='%H:%M:%S:%f').dt.strftime('%H:%M')
#Date formatting error
s123_divedata_dbf_reset['Dive Date'] = s123_divedata_dbf_reset['Dive Date'].apply(lambda x: datetime.strptime(x, "%m/%d/%Y").strftime("%Y-%m-%d"))
#FloridaAtlantic sync error
s123_divedata_dbf_reset['Location'] = s123_divedata_dbf_reset['Location'].replace("FloridaAtlantic", "Florida Atlantic")
#Mission Iconic Reef sync error
s123_divedata_dbf_reset['Project'] = s123_divedata_dbf_reset['Project'].replace("M:IR", "Mission: Iconic Reefs")
#BOEM - Carolina Long Bay sync error
s123_divedata_dbf_reset['Project'] = s123_divedata_dbf_reset['Project'].replace("BOEM-CLB", "BOEM: Carolina Long Bay")

#s123_divedata_dbf_reset.head()

In [ ]:
#Isolate out the global IDs from s123 so that it can be compared with SMartsheet to avoid duplicates
s123_divedata_globalid = s123_divedata_dbf_reset['globalid'].tolist()
print(s123_divedata_globalid)

Start working in Smartsheet

In [178]:
# Set up Smartsheet access
access_token = '6H72JtOX0pVZaTumEofA0zla9X1stAruyhHBz' #'--PUT ACCESS TOKEN HERE--'
smartSheet_Client = smartsheet.Smartsheet(access_token, api_base=smartsheet.__gov_base__)
smartSheet_Client.errors_as_exceptions(True)
divelog_sheetid = '3136216628520844'

# Function to get the maximum Edit Date from Smartsheet
def get_max_edit_date(sheet, edit_date_column_id):
    max_edit_date = None
    for row in sheet.rows:
        for cell in row.cells:
            if cell.column_id == edit_date_column_id and cell.value:
                edit_date = datetime.strptime(cell.value, '%Y-%m-%dT%H:%M:%SZ').date()
                if not max_edit_date or edit_date > max_edit_date:
                    max_edit_date = edit_date
    return max_edit_date

In [ ]:
#find the most recent smartsheet edit
client = smartSheet_Client
sheet = client.Sheets.get_sheet(divelog_sheetid)
column_id_map = get_column_id_map(sheet)
edit_date_column_id = column_id_map.get('Edit Date')

if not edit_date_column_id:
    print("Edit Date column not found.")

max_edit_date = get_max_edit_date(sheet, edit_date_column_id)
if not max_edit_date:
    print("No edit date found in the Smartsheet.")

print(f"Max Edit Date: {max_edit_date}")

In [ ]:
#make a list of all smartsheet global ids to compare with survey123 to avoid duplicates
COLUMN_NAME = "globalid"  # Replace with the desired column name

# Find the column ID based on the column name
column_id = None
for column in sheet.columns:
    if column.title == COLUMN_NAME:
        column_id = column.id
        break

if column_id is None:
    print(f"Column '{COLUMN_NAME}' not found in the sheet.")
else:
    # Extract populated values from the specified column
    smartsheet_divedata_globalids = []
    for row in sheet.rows:
        for cell in row.cells:
            if cell.column_id == column_id and cell.value is not None:
                smartsheet_divedata_globalids.append(cell.value)
                break  # Move to the next row after finding a populated value

    # Display the populated values list (you can now compare this list with another)
    print(f"Populated values in column '{COLUMN_NAME}':")
    print(smartsheet_divedata_globalids)

In [ ]:
#compare lists and cut duplicates
newdivedata = s123_divedata_dbf_reset[~s123_divedata_dbf_reset['globalid'].isin(smartsheet_divedata_globalids)]
print(newdivedata)

In [182]:
## Functions to complete the processing of the dataframe and upload to Smartsheet

# Function to get the column ID map
def get_column_id_map(sheet):
    return {col.title: col.id for col in sheet.columns}

# Function to prepare Smartsheet rows
def prepare_smartsheet_rows(data, column_id_map, extra_columns):
    new_rows = []
    for _, item in data.iterrows():  # Use iterrows to loop through DataFrame rows
        row = smartsheet.models.Row()
        row.to_bottom = True
        for column, value in item.items():
            if column in column_id_map and pd.notna(value):  # Check for missing values
                row.cells.append({'column_id': column_id_map[column], 'value': value})
            elif column in column_id_map:
                row.cells.append({'column_id': column_id_map[column], 'value': ''})  # Use an empty string if NA
        if extra_columns:
            for column, value in extra_columns.items():
                row.cells.append({'column_id': column_id_map[column], 'value': value})
        new_rows.append(row)
    return new_rows

# Function to add rows to Smartsheet
def add_rows_to_smartsheet(client, sheet_id, rows):
    try:
        response = client.Sheets.add_rows(sheet_id, rows)
        print(f"Uploaded {len(rows)} rows successfully!")
    except smartsheet.exceptions.ApiError as e:
        print(f"Error adding rows to Smartsheet: {e}")

# Main function to process each sheet
def process_sheet(client, sheet_info, data):
    # Retrieve the sheet
    sheet = client.Sheets.get_sheet(sheet_info['sheet_id'])
    # Get the column ID map
    column_id_map = get_column_id_map(sheet)

    # Prepare rows for Smartsheet
    rows = prepare_smartsheet_rows(data, column_id_map, sheet_info.get('extra_columns'))
    # Add rows to Smartsheet
    add_rows_to_smartsheet(client, sheet_info['sheet_id'], rows)

# Define the information for each sheet
sheets_info = [
    {
        'sheet_id': '3136216628520844',
        'fields_map': {
            'Project': 'Project',
            'Mission': 'Mission',
            'Location': 'Location',
            'Platform': 'Platform',
            'Vehicle': 'Vehicle',
            'Dive Date': 'Dive Date',
            'Dive Number': 'Dive Number',
            'Site': 'Site',
            'Mission Start Time': 'Mission Start Time',
            'Start Latitude': 'Start Latitude',
            'Start Longitude': 'Start Longitude',
            'End Time': 'End Time',
            'Dive Description': 'Dive Description',
            'globalid': 'globalid'
        }
    }
]

# Run the processing for each sheet
for sheet_info in sheets_info:
    # Map DataFrame columns based on 'fields_map'
    data_to_upload = newdivedata[list(sheet_info['fields_map'].keys())]
    data_to_upload.rename(columns=sheet_info['fields_map'], inplace=True)
    process_sheet(smartSheet_Client, sheet_info, data_to_upload)


C:\Users\Mike.Bollinger\AppData\Local\Temp\1\ipykernel_10636\2581601327.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_to_upload.rename(columns=sheet_info['fields_map'], inplace=True)


Uploaded 0 rows successfully!
